# Fig4 — Microglia Subtypes & WGCNA Modules
**Panel A**: UMAP of microglia subtypes (Python / scanpy).
**Panel B**: Functional cluster marker heatmap (R / ComplexHeatmap).
**Panel C**: Developmental stage proportions (R / ggplot2).
**Panel D**: Function cluster × brain region enrichment heatmap (R).
**Panel E**: WGCNA module dotplot by function cluster (R / Seurat).
**Panel F**: WGCNA module trajectory across development (R / hdWGCNA).


## Panel A: UMAP of Microglia Subtypes (Python / scanpy)


In [ ]:
import os
import scanpy as sc
from matplotlib.pyplot import rc_context

proj_dir = os.environ.get('PROJ_DIR', 'path/to/your/project')
data_dir = os.path.join(proj_dir, 'data', 'human', 'output')
out_dir  = os.path.join(data_dir, 'Visualization', 'Fig4')
os.makedirs(out_dir, exist_ok=True)

# Load microglia AnnData
mic_path = os.path.join(data_dir, 'Annotation_hvg5000', 'L4', 'hs_anno_4_Mic.h5ad')
ad_hs = sc.read_h5ad(mic_path)

# Recompute UMAP if needed
sc.pp.neighbors(ad_hs, n_pcs=50, n_neighbors=60)
sc.tl.umap(ad_hs, min_dist=0.3, spread=1.0, random_state=123, maxiter=500)

# Colour palette
type_3_color = {
    "Mic-1": "#B8E2F0", "Mic-2": "#B890BD", "Mic-3": "#F9CC70",
    "Mic-4": "#FDE4AA", "Mic-5": "#FBD980", "Mic-6": "#AAC5E0",
    "Mic-7": "#F7B450", "Mic-8": "#F8C060", "Mic-9": "#F8B4BA",
    "Mic-10": "#92639C", "Mic-11": "#7F9BC8", "Mic-12": "#80508A",
    "Mic-13": "#73C79E", "Mic-14": "#6E3C78", "Mic-15": "#85CBD8",
    "Mic-16": "#F27E88", "Mic-17": "#F5A840"
}
ad_hs.uns["type_3_abbr_colors"] = [type_3_color.get(c, "#CCCCCC") for c in ad_hs.obs["type_3_abbr"].cat.categories]


In [ ]:
# Draw and save UMAP
sc.settings.set_figure_params(dpi=300, facecolor="white")
with rc_context({"figure.figsize": (6, 5)}):
    sc.pl.umap(ad_hs, color="type_3_abbr", size=10,
              legend_loc="right margin", palette=type_3_color,
              legend_fontsize=10, legend_fontoutline=2,
              save=os.path.join(out_dir, "mic_type_3_umap.pdf"))
print("Panel A saved")

## Panels B–F: R (Seurat / ComplexHeatmap / hdWGCNA)


In [ ]:
%%R
library(Seurat)
library(ComplexHeatmap)
library(circlize)
library(ggplot2)
library(dplyr)
library(tidyr)
library(purrr)
library(patchwork)
library(RColorBrewer)
library(grid)


In [ ]:
%%R
# ============================================================================
# Configuration


In [ ]:
%%R
# ============================================================================
proj_dir <- Sys.getenv("PROJ_DIR", unset = "path/to/your/project")  # <-- MODIFY THIS
setwd(file.path(proj_dir, "data", "human", "output"))

outdir <- "Visualization/Fig4"
dir.create(outdir, recursive = TRUE, showWarnings = FALSE)
ct <- "Mic"


In [ ]:
%%R
# ============================================================================
# Shared colour palettes


In [ ]:
%%R
# ============================================================================
type_3_color <- c(
  "Mic-1" = "#B8E2F0", "Mic-2" = "#B890BD", "Mic-3" = "#F9CC70",
  "Mic-4" = "#FDE4AA", "Mic-5" = "#FBD980", "Mic-6" = "#AAC5E0",
  "Mic-7" = "#F7B450", "Mic-8" = "#F8C060", "Mic-9" = "#F8B4BA",
  "Mic-10" = "#92639C", "Mic-11" = "#7F9BC8", "Mic-12" = "#80508A",
  "Mic-13" = "#73C79E", "Mic-14" = "#6E3C78", "Mic-15" = "#85CBD8",
  "Mic-16" = "#F27E88", "Mic-17" = "#F5A840"
)

function_cluster_colors <- c(
  "Proliferative" = "#D175A9", "Enhanced redox" = "#A09CBF",
  "Inflammatory" = "#C47401", "Lipid associated" = "#4C949F",
  "Surveilling" = "#2A71AA", "Reacting" = "grey"
)

region_l1_colors <- c(
  "Brain" = "#5c8e9b", "Diencephalon" = "#85b76e",
  "Mesencephalon" = "#decf98", "Metencephalon " = "#874e94",
  "Myelencephalon " = "#e97c17", "Prosencephalon" = "#da84b6",
  "Rhombencephalon" = "#4f9446", "Spinal cord" = "#2a71aa",
  "Telencephalon" = "#ae94d7"
)


In [ ]:
%%R
# ============================================================================
# Load data


In [ ]:
%%R
# ============================================================================
obj <- readRDS(paste0("Analysis/immune/validation/", ct, "/seurat_obj_function.rds"))


In [ ]:
%%R
# ============================================================================
# Panel A: UMAP of microglia subtypes
# ============================================================================
obj_a <- obj
Idents(obj_a) <- "type_3_abbr"

pdf(file.path(outdir, "mic_type_3_umap.pdf"), width = 6, height = 5)
DimPlot(obj_a, reduction = "umap", group.by = "type_3_abbr",
        cols = type_3_color, pt.size = 0.5, label = FALSE) +
  NoLegend() + ggtitle("") +
  theme(plot.background = element_rect(fill = "white", color = NA),
        panel.background = element_rect(fill = "white"))
dev.off()
message("Panel A saved")


In [ ]:
%%R
# ============================================================================
# Panel B: Functional cluster marker heatmap
# ============================================================================
markers_file <- paste0("Analysis/immune/validation/", ct,
                       "/mic_functional_cluster6_marker_new.csv")
all_markers <- read.csv(markers_file)

cluster_order <- c("Proliferative", "Enhanced redox", "Inflammatory",
                   "Lipid associated", "Surveilling", "Reacting")

top_markers <- all_markers %>%
  filter(p_val_adj < 0.05, avg_log2FC > 0) %>%
  group_by(cluster) %>%
  top_n(30, avg_log2FC) %>%
  ungroup() %>%
  mutate(cluster = factor(cluster, levels = cluster_order)) %>%
  arrange(cluster, desc(avg_log2FC))

genes_plot <- unique(top_markers$gene)

# Average expression per function cluster
expr_mat <- AverageExpression(
  obj, assays = "RNA", slot = "data",
  features = genes_plot, group.by = "function_cluster"
)$RNA
expr_mat <- expr_mat[genes_plot, cluster_order]
expr_mat <- t(scale(t(expr_mat)))
expr_mat[is.na(expr_mat)] <- 0
expr_mat[expr_mat > 2.5] <- 2.5
expr_mat[expr_mat < -2.5] <- -2.5

fill_col <- colorRamp2(c(-2.5, 0, 2.5), c("#2166ac", "#f7f7f7", "#b2182b"))

# Row annotation: cluster membership
gene_cluster <- top_markers$cluster[match(genes_plot, top_markers$gene)]
row_annotation <- rowAnnotation(
  Cluster = gene_cluster,
  col = list(Cluster = function_cluster_colors),
  show_legend = FALSE, simple_anno_size = unit(0.15, "cm")
)

ht_b <- Heatmap(
  expr_mat, name = "Scaled Expression", col = fill_col,
  top_annotation = HeatmapAnnotation(
    Cluster = factor(cluster_order, levels = cluster_order),
    col = list(Cluster = function_cluster_colors),
    show_legend = FALSE, simple_anno_size = unit(0.15, "cm")
  ),
  left_annotation = row_annotation,
  show_row_names = FALSE, show_column_names = FALSE,
  cluster_rows = FALSE, cluster_columns = FALSE,
  column_split = factor(cluster_order, levels = cluster_order),
  column_title = NULL, use_raster = TRUE,
  heatmap_legend_param = list(title = "Scaled Expression", direction = "horizontal")
)

pdf(file.path(outdir, "functional_cluster6_heatmap.pdf"), width = 8, height = 18)
draw(ht_b, heatmap_legend_side = "right", annotation_legend_side = "right")
dev.off()
message("Panel B saved")


In [ ]:
%%R
# ============================================================================
# Panel C: Developmental stage distribution of microglia subtypes
# ============================================================================
stage_dist <- obj@meta.data %>%
  filter(!is.na(type_3_abbr)) %>%
  count(type_3_abbr, stage) %>%
  group_by(type_3_abbr) %>%
  mutate(proportion = n / sum(n)) %>%
  ungroup()

p_c <- ggplot(stage_dist, aes(x = stage, y = proportion, fill = type_3_abbr)) +
  geom_col(position = "fill") +
  scale_fill_manual(values = type_3_color) +
  labs(x = "Developmental Stage", y = "Proportion",
       title = "Developmental Stage Distribution of Microglia Subtypes") +
  theme_minimal() +
  theme(
    axis.text.x = element_blank(), axis.ticks.x = element_blank(),
    legend.position = "right",
    plot.title = element_text(hjust = 0.5, size = 14, face = "bold"),
    plot.background = element_rect(fill = "white", color = NA),
    panel.background = element_rect(fill = "white")
  )

ggsave(file.path(outdir, "type_3_developmental_stage.pdf"),
       p_c, width = 12, height = 6)
message("Panel C saved")


In [ ]:
%%R
# ============================================================================
# Panel D: Function cluster × brain region enrichment heatmap
# ============================================================================
total_cells <- nrow(obj@meta.data)

region_totals <- obj@meta.data %>%
  count(region_L2) %>% dplyr::rename(total_region = n)
type_totals <- obj@meta.data %>%
  count(function_cluster) %>% dplyr::rename(total_type = n)

enrich_data <- obj@meta.data %>%
  count(function_cluster, region_L2) %>%
  left_join(region_totals, by = "region_L2") %>%
  left_join(type_totals, by = "function_cluster") %>%
  mutate(
    a = n, b = total_type - n, c = total_region - n,
    d = total_cells - a - b - c,
    or_val = (a / b) / (c / d),
    or_val = ifelse(is.infinite(or_val) | is.na(or_val), 1, or_val)
  )

enrich_mat <- enrich_data %>%
  select(function_cluster, region_L2, or_val) %>%
  pivot_wider(names_from = region_L2, values_from = or_val, values_fill = 1) %>%
  as.data.frame()
rownames(enrich_mat) <- enrich_mat$function_cluster
enrich_mat$function_cluster <- NULL
enrich_mat <- as.matrix(enrich_mat[cluster_order, ])
enrich_mat <- log2(enrich_mat)

# Region L1 annotation
region_mapping <- obj@meta.data %>%
  select(region_L2, region_L1) %>% distinct()
colnames(enrich_mat) -> rnames
l1_info <- region_mapping$region_L1[match(rnames, region_mapping$region_L2)]
region_order <- order(l1_info)
enrich_mat <- enrich_mat[, region_order, drop = FALSE]
l1_ordered <- l1_info[region_order]

top_anno <- HeatmapAnnotation(
  Region_L1 = l1_ordered,
  col = list(Region_L1 = region_l1_colors),
  which = "column", show_legend = TRUE,
  annotation_name_side = "left", simple_anno_size = unit(0.2, "cm")
)
left_anno <- rowAnnotation(
  Function = factor(cluster_order, levels = cluster_order),
  col = list(Function = function_cluster_colors),
  show_legend = FALSE, simple_anno_size = unit(0.2, "cm")
)

col_fun_d <- colorRamp2(
  seq(min(enrich_mat), max(enrich_mat), length.out = 5),
  c("white", "#F6BDC0", "#F1959B", "#EA4C46", "#781426")
)

ht_d <- Heatmap(
  enrich_mat, name = "log2(OR)", col = col_fun_d,
  bottom_annotation = top_anno, left_annotation = left_anno,
  cluster_rows = FALSE, cluster_columns = FALSE,
  show_row_names = TRUE, show_column_names = TRUE,
  column_names_rot = 90, row_names_side = "left",
  rect_gp = gpar(col = "white", lwd = 0.5),
  border = TRUE, border_gp = gpar(col = "black", lwd = 1),
  heatmap_legend_param = list(title = "log2(OR)", direction = "horizontal")
)

pdf(file.path(outdir, "function_cluster_region_l2_heatmap.pdf"), width = 12, height = 6)
draw(ht_d, annotation_legend_side = "right", heatmap_legend_side = "right",
     merge_legend = TRUE)
dev.off()
message("Panel D saved")


In [ ]:
%%R
# ============================================================================
# Panel E & F: WGCNA module dotplot + trajectory
# ============================================================================
wgcna_file <- paste0("Analysis/WGCNA/", ct, "/microglia_hdWGCNA_obj.rds")
if (file.exists(wgcna_file)) {
  obj_wgcna <- readRDS(wgcna_file)

  mods <- GetModules(obj_wgcna) %>%
    filter(module != "grey") %>%
    pull(module) %>% unique() %>% as.character()

  # Panel E: Module × function cluster dotplot
  p_e <- DotPlot(obj_wgcna, features = mods, group.by = "function_cluster",
                 dot.scale = 10) +
    coord_flip() +
    scale_color_gradientn(colors = rev(brewer.pal(11, "RdYlBu"))) +
    theme(
      axis.text.x = element_text(size = 14, angle = 0),
      axis.text.y = element_text(size = 16),
      plot.background = element_rect(fill = "white", color = NA),
      panel.background = element_rect(fill = "white")
    )

  pdf(file.path(outdir, "ModuleExpression_functioncluster_DotPlot.pdf"),
      width = 10, height = 6)
  print(p_e)
  dev.off()
  message("Panel E saved")

  # Panel F: Module trajectory across developmental stages
  stage_mapping <- c(
    "01-Embryonic" = 1, "02-Early fetal" = 2, "03-Early fetal" = 3,
    "04-Early midfetal" = 4, "05-Early midfetal" = 5, "06-Late midfetal" = 6,
    "07-Late fetal" = 7, "08-Neonatal and early infancy" = 8,
    "09-Late infancy" = 9, "10-Early childhood" = 10,
    "11-Middle and late childhood" = 11, "12-Adolescence" = 12,
    "13-Young adulthood" = 13, "14-Middle adulthood" = 14,
    "15-Late adulthood" = 15
  )
  obj_wgcna@meta.data$stage_num <- stage_mapping[obj_wgcna@meta.data$stage]

  p_f <- PlotModuleTrajectory(obj_wgcna, pseudotime_col = "stage_num", ncol = 2) +
    theme(plot.background = element_rect(fill = "white", color = NA),
          panel.background = element_rect(fill = "white"))

  ggsave(file.path(outdir, "Module_Trajectory.pdf"), p_f, width = 10, height = 4)
  message("Panel F saved")
} else {
  message("WGCNA RDS not found, skipping panels E–F: ", wgcna_file)
}

message("Fig4 complete. All panels saved to: ", outdir)
